# M8_8.31–M8_8.33 · Funciones, automatización y reproducibilidad

Este cuaderno continúa el flujo desarrollado desde M8_8.19:

```text
Python básico → NumPy → pandas → Matplotlib → SQLite → estadística y calidad
→ funciones → automatización → flujo reproducible
```

Se utilizan los mismos archivos compartidos del repositorio. No se genera un conjunto de datos alternativo ni una segunda estructura de proyecto.

## Índice

### M8_8.31 · Funciones y organización
- Código exploratorio y código reutilizable
- Responsabilidad de una función
- Parámetros, argumentos y valores devueltos
- Funciones puras y efectos secundarios
- Docstrings, validación y errores
- Pruebas sencillas con `assert`
- Scripts, módulos y `main`

### M8_8.32 · Automatización y control
- Automatizar una operación conocida
- Configuración separada de la lógica
- Rutas portables
- Procesamiento de varios pozos
- Errores esperados
- Registro con `logging`
- Idempotencia
- Guardado de tablas y figuras

### M8_8.33 · Flujo reproducible
- Carga desde SQLite
- Validación
- Transformación
- Resumen y figuras
- Registro de ejecución
- Manifiesto de resultados
- Ejecución completa y comprobaciones finales

## Preparación: localizar el repositorio

El cuaderno espera la estructura utilizada en M8_8.22–27. En Google Colab debe clonarse primero el repositorio del curso.

In [ ]:
from pathlib import Path
import datetime
import json
import logging
import platform
import sqlite3
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def encontrar_raiz_repositorio():
    """Busca hacia arriba una carpeta que contenga data/input."""
    carpeta_actual = Path.cwd().resolve()

    for carpeta in [carpeta_actual, *carpeta_actual.parents]:
        if (carpeta / "data" / "input").exists():
            return carpeta

    raise FileNotFoundError(
        "No se encuentra data/input. En Colab, clona primero el repositorio."
    )


ROOT = encontrar_raiz_repositorio()
INPUT = ROOT / "data" / "input"
DATABASE = ROOT / "data" / "database"
OUTPUT = ROOT / "data" / "output"
TABLES = OUTPUT / "tables"
FIGURES = OUTPUT / "figures"
LOGS = OUTPUT / "logs"

for carpeta in [TABLES, FIGURES, LOGS]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Raíz del repositorio:", ROOT)

## Cargar los mismos datos desde SQLite

En M8_8.25 se creó `hidrogeologia.sqlite` a partir de los CSV compartidos. Aquí se recupera una consulta ya combinada como DataFrame. La consulta SQL selecciona los campos; pandas continúa el análisis.

In [ ]:
ruta_base_datos = DATABASE / "hidrogeologia.sqlite"

consulta = """
SELECT
    m.id_pozo,
    m.fecha,
    p.acuifero,
    p.cota_terreno_m,
    m.profundidad_nivel_m,
    m.precipitacion_mm,
    m.conductividad_uScm
FROM mediciones AS m
INNER JOIN pozos AS p
    ON m.id_pozo = p.id_pozo
ORDER BY m.id_pozo, m.fecha;
"""

with sqlite3.connect(ruta_base_datos) as conexion:
    datos = pd.read_sql_query(
        consulta,
        conexion,
        parse_dates=["fecha"]
    )

print("Dimensiones:", datos.shape)
display(datos.head())

---
# M8_8.31 · Funciones y organización del código

## 1. De código exploratorio a código reutilizable

El código exploratorio ayuda a probar una idea. El código reutilizable expresa una operación mediante una función con nombre, entradas y salida definidos.

El ejemplo repetido funciona, pero obliga a copiar y modificar código. Esto aumenta el riesgo de inconsistencias.

In [ ]:
# Código repetido: útil para observar el problema, no como solución final.
p01 = datos.loc[datos["id_pozo"] == "P01"]
media_p01 = p01["profundidad_nivel_m"].mean()
print("P01:", media_p01)

p02 = datos.loc[datos["id_pozo"] == "P02"]
media_p02 = p02["profundidad_nivel_m"].mean()
print("P02:", media_p02)

## 2. Anatomía de una función

```python
def nombre_funcion(parametro):
    resultado = ...
    return resultado
```

- **Nombre:** describe la acción.
- **Parámetro:** nombre utilizado en la definición.
- **Argumento:** valor proporcionado al llamar la función.
- **Cuerpo:** instrucciones indentadas.
- **`return`:** devuelve un resultado reutilizable.

Una función no necesita ser corta a cualquier precio. Debe ser comprensible y tener una responsabilidad reconocible.

In [ ]:
def seleccionar_pozo(datos_entrada, id_pozo):
    """Devuelve una copia de las observaciones correspondientes a un pozo."""

    ids_disponibles = set(datos_entrada["id_pozo"])

    if id_pozo not in ids_disponibles:
        raise ValueError(f"Pozo desconocido: {id_pozo}")

    es_pozo_solicitado = datos_entrada["id_pozo"] == id_pozo
    datos_pozo = datos_entrada.loc[es_pozo_solicitado].copy()

    return datos_pozo


p01 = seleccionar_pozo(datos, "P01")
display(p01.head())

## 3. Docstrings y contrato de una función

Una **docstring** explica qué hace la función. Para que sirva como referencia, conviene documentar parámetros, resultado y errores esperados.

El **contrato** de una función describe:

- qué entradas acepta;
- qué condiciones deben cumplir;
- qué devuelve;
- qué errores puede producir;
- si modifica objetos o guarda archivos.

In [ ]:
def calcular_cota_piezometrica(cota_terreno_m, profundidad_nivel_m):
    """Calcula una cota piezométrica.

    Parameters
    ----------
    cota_terreno_m : float
        Cota del terreno expresada en metros.
    profundidad_nivel_m : float
        Profundidad del nivel medida desde la superficie, en metros.

    Returns
    -------
    float
        Cota piezométrica en metros.

    Raises
    ------
    ValueError
        Si la profundidad es negativa.
    """

    if profundidad_nivel_m < 0:
        raise ValueError("La profundidad del nivel no puede ser negativa.")

    resultado = cota_terreno_m - profundidad_nivel_m
    return resultado


cota = calcular_cota_piezometrica(112.4, 8.7)
print("Cota piezométrica:", cota, "m")

## 4. Parámetros y valores por defecto

Los parámetros evitan valores ocultos dentro del código. Un valor por defecto es útil cuando existe una opción habitual, pero debe poder modificarse explícitamente.

In [ ]:
def resumir_variable(datos_entrada, variable, incluir_desviacion=True):
    """Devuelve un resumen descriptivo de una columna numérica."""

    if variable not in datos_entrada.columns:
        raise KeyError(f"No existe la columna: {variable}")

    serie = pd.to_numeric(
        datos_entrada[variable],
        errors="coerce"
    )

    resumen = {
        "numero_validos": serie.count(),
        "media": serie.mean(),
        "mediana": serie.median(),
        "minimo": serie.min(),
        "maximo": serie.max()
    }

    if incluir_desviacion:
        resumen["desviacion_estandar"] = serie.std()

    return pd.Series(resumen)


resumen_p01 = resumir_variable(
    datos_entrada=p01,
    variable="profundidad_nivel_m"
)

display(resumen_p01)

## 5. `return` frente a `print`

`print` comunica algo en pantalla. `return` entrega un objeto al resto del programa.

Una función analítica suele devolver el resultado. Así puede mostrarse, guardarse, combinarse o comprobarse posteriormente.

In [ ]:
def media_que_imprime(serie):
    print(serie.mean())


def media_que_devuelve(serie):
    resultado = serie.mean()
    return resultado


resultado_print = media_que_imprime(p01["profundidad_nivel_m"])
resultado_return = media_que_devuelve(p01["profundidad_nivel_m"])

print("Valor devuelto por la primera función:", resultado_print)
print("Valor devuelto por la segunda función:", resultado_return)

## 6. Funciones puras y efectos secundarios

Una **función pura** calcula una salida a partir de entradas y no modifica el exterior. Es más fácil de probar.

Un **efecto secundario** cambia algo fuera de la función, por ejemplo:

- guardar un archivo;
- modificar un DataFrame recibido;
- escribir un registro;
- mostrar una figura.

Los efectos secundarios no son incorrectos. Deben ser visibles y separarse, cuando sea posible, de la lógica analítica.

In [ ]:
def añadir_cota_piezometrica(datos_entrada):
    """Devuelve una copia con una columna de cota piezométrica."""

    resultado = datos_entrada.copy()

    resultado["cota_piezometrica_m"] = (
        resultado["cota_terreno_m"]
        - resultado["profundidad_nivel_m"]
    )

    return resultado


datos_con_cota = añadir_cota_piezometrica(datos)

print("¿La columna existe en el original?", "cota_piezometrica_m" in datos.columns)
print("¿La columna existe en la copia?", "cota_piezometrica_m" in datos_con_cota.columns)

## 7. Validación antes del cálculo

Validar significa comprobar requisitos. Una validación puede:

- detener el flujo si falta una condición esencial;
- devolver un informe de incidencias;
- marcar observaciones para revisión.

No toda incidencia exige eliminar datos.

In [ ]:
def validar_mediciones(datos_entrada):
    """Comprueba estructura básica y devuelve un informe de calidad."""

    columnas_requeridas = {
        "id_pozo",
        "fecha",
        "profundidad_nivel_m"
    }

    columnas_presentes = set(datos_entrada.columns)
    columnas_ausentes = columnas_requeridas - columnas_presentes

    if columnas_ausentes:
        raise ValueError(
            f"Faltan columnas requeridas: {sorted(columnas_ausentes)}"
        )

    informe = {
        "numero_filas": len(datos_entrada),
        "numero_pozos": datos_entrada["id_pozo"].nunique(),
        "niveles_ausentes": int(
            datos_entrada["profundidad_nivel_m"].isna().sum()
        ),
        "filas_duplicadas": int(
            datos_entrada.duplicated().sum()
        ),
        "profundidades_negativas": int(
            (datos_entrada["profundidad_nivel_m"] < 0).sum()
        )
    }

    return informe


informe_validacion = validar_mediciones(datos)
print(informe_validacion)

## 8. Pruebas sencillas con `assert`

Una **prueba** comprueba que una operación produce el comportamiento esperado. `assert` detiene la ejecución si una condición es falsa.

Estas pruebas no sustituyen un sistema completo de testing, pero ayudan a detectar cambios inesperados.

In [ ]:
# Caso conocido
resultado_conocido = calcular_cota_piezometrica(100.0, 8.0)
assert resultado_conocido == 92.0

# La selección debe contener un único identificador.
seleccion = seleccionar_pozo(datos, "P01")
assert seleccion["id_pozo"].nunique() == 1
assert seleccion["id_pozo"].iloc[0] == "P01"

# La función no debe modificar el DataFrame original.
assert "cota_piezometrica_m" not in datos.columns

print("Pruebas superadas.")

## 9. Script, módulo y `main`

Las funciones reutilizables pueden guardarse en `src/analisis.py`. Otro archivo puede importarlas.

```python
from src.analisis import seleccionar_pozo
```

El patrón siguiente delimita la ejecución principal:

```python
def main():
    datos = cargar_datos(...)
    resultado = ejecutar_flujo(datos, ...)
    print(resultado)

if __name__ == "__main__":
    main()
```

Cuando el archivo se ejecuta directamente, `__name__` vale `"__main__"`. Cuando se importa como módulo, el bloque no se ejecuta automáticamente.

### Actividad M8_8.31

1. Convierte una operación repetida en una función.
2. Añade una docstring con parámetros, resultado y errores.
3. Separa cálculo y guardado de archivos.
4. Valida una columna requerida.
5. Escribe dos pruebas con `assert`.
6. Explica qué parte guardarías en `src/analisis.py`.

---
# M8_8.32 · Automatización y controles

## 10. Qué significa automatizar

Automatizar no significa únicamente añadir un bucle. Significa aplicar un procedimiento definido de manera consistente a varias unidades, registrar qué ocurrió y producir resultados trazables.

La automatización es apropiada cuando la tarea ya se entiende. Automatizar un procedimiento incorrecto solo repite el error más rápidamente.

## 11. Configuración separada de la lógica

La configuración contiene decisiones que pueden cambiar entre ejecuciones: variable, pozos, formato y resolución. La función contiene la lógica estable.

JSON es un formato legible y compatible con muchos lenguajes.

In [ ]:
configuracion = {
    "variable": "profundidad_nivel_m",
    "pozos": ["P01", "P02", "P03"],
    "dpi": 150,
    "formato_figura": "png"
}

ruta_configuracion = LOGS / "configuracion_31_33.json"

ruta_configuracion.write_text(
    json.dumps(
        configuracion,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

configuracion_leida = json.loads(
    ruta_configuracion.read_text(encoding="utf-8")
)

print(configuracion_leida)

## 12. Automatizar resúmenes por pozo

El bucle coordina funciones ya definidas. Cada paso se mantiene explícito para facilitar el diagnóstico.

In [ ]:
resumenes = []

for id_pozo in configuracion_leida["pozos"]:
    datos_pozo = seleccionar_pozo(
        datos_entrada=datos,
        id_pozo=id_pozo
    )

    resumen_pozo = resumir_variable(
        datos_entrada=datos_pozo,
        variable=configuracion_leida["variable"]
    )

    resumen_pozo["id_pozo"] = id_pozo
    resumenes.append(resumen_pozo)

resumen_todos = pd.DataFrame(resumenes)
resumen_todos = resumen_todos.set_index("id_pozo")

display(resumen_todos)

## 13. Guardar una tabla de resultados

Guardar es un efecto secundario. La función devuelve la ruta para que el flujo pueda registrarla o comprobarla.

In [ ]:
def guardar_dataframe_csv(dataframe, ruta):
    """Guarda un DataFrame como CSV y devuelve la ruta creada."""

    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)

    dataframe.to_csv(
        ruta,
        index=True
    )

    return ruta


ruta_resumen = guardar_dataframe_csv(
    dataframe=resumen_todos,
    ruta=TABLES / "resumen_pozos.csv"
)

print("Tabla guardada en:", ruta_resumen)

## 14. Función para guardar una figura

La función recibe datos ya seleccionados. No decide qué pozo analizar. Esta separación evita mezclar selección, análisis y salida.

In [ ]:
def guardar_serie_temporal(datos_pozo, ruta, dpi=150):
    """Guarda una serie temporal de profundidad y devuelve su ruta."""

    if datos_pozo.empty:
        raise ValueError("No hay observaciones para representar.")

    datos_ordenados = datos_pozo.sort_values("fecha")
    id_pozo = datos_ordenados["id_pozo"].iloc[0]

    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)

    figura, eje = plt.subplots(figsize=(8, 4))

    eje.plot(
        datos_ordenados["fecha"],
        datos_ordenados["profundidad_nivel_m"],
        marker="o"
    )

    eje.invert_yaxis()
    eje.set_title(f"Profundidad del nivel en {id_pozo}")
    eje.set_xlabel("Fecha")
    eje.set_ylabel("Profundidad del nivel (m)")

    figura.autofmt_xdate()
    figura.tight_layout()

    figura.savefig(
        ruta,
        dpi=dpi,
        bbox_inches="tight"
    )

    plt.close(figura)
    return ruta


ruta_figura_p01 = guardar_serie_temporal(
    datos_pozo=seleccionar_pozo(datos, "P01"),
    ruta=FIGURES / "P01_profundidad_nivel.png",
    dpi=configuracion_leida["dpi"]
)

print("Figura guardada en:", ruta_figura_p01)

## 15. Errores esperados con `try/except`

`try/except` se usa cuando conocemos una excepción que podemos gestionar. No debe capturar todo indiscriminadamente ni convertir errores en silencio.

En este ejemplo, un pozo inexistente se registra y el procesamiento continúa.

In [ ]:
ids_para_probar = ["P01", "P99", "P02"]

for id_pozo in ids_para_probar:
    try:
        datos_pozo = seleccionar_pozo(datos, id_pozo)
        print(id_pozo, "→", len(datos_pozo), "filas")

    except ValueError as error:
        print(id_pozo, "→ ERROR:", error)

## 16. Registro con `logging`

Un registro informa de lo ocurrido sin mezclarse con el resultado analítico.

Niveles habituales:

- `INFO`: paso normal del flujo;
- `WARNING`: incidencia que permite continuar;
- `ERROR`: operación que no pudo completarse.

Un archivo de log ayuda a revisar una ejecución automática.

In [ ]:
ruta_log = LOGS / "ejecucion_31_33.log"

logger = logging.getLogger("flujo_hidrogeologico")
logger.setLevel(logging.INFO)
logger.handlers.clear()

manejador_archivo = logging.FileHandler(
    ruta_log,
    mode="w",
    encoding="utf-8"
)

formato = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

manejador_archivo.setFormatter(formato)
logger.addHandler(manejador_archivo)

logger.info("Inicio del registro de ejemplo")
logger.info("Número de filas: %s", len(datos))
logger.warning("Las incidencias de calidad deben revisarse")

print("Registro creado en:", ruta_log)

## 17. Idempotencia

Un proceso es **idempotente** cuando repetirlo con las mismas entradas y configuración produce el mismo estado final esperado.

Ejemplos:

- crear carpetas con `exist_ok=True`;
- sobrescribir un resultado regenerable con el mismo nombre;
- no añadir las mismas filas una y otra vez;
- fijar una semilla si existe aleatoriedad.

La idempotencia permite ejecutar de nuevo el notebook sin limpiar manualmente resultados anteriores.

In [ ]:
# Ejecutar estas instrucciones varias veces no crea carpetas duplicadas.
for carpeta in [TABLES, FIGURES, LOGS]:
    carpeta.mkdir(parents=True, exist_ok=True)

# El mismo archivo regenerable se sobrescribe de forma controlada.
ruta_comprobacion = TABLES / "comprobacion_idempotencia.csv"

pd.DataFrame({
    "estado": ["correcto"]
}).to_csv(
    ruta_comprobacion,
    index=False
)

print("Resultado regenerado:", ruta_comprobacion)

### Actividad M8_8.32

1. Crea una configuración con dos pozos y una variable.
2. Automatiza el resumen de ambos pozos.
3. Guarda una tabla y dos figuras.
4. Gestiona un identificador no existente.
5. Registra inicio, final e incidencias.
6. Explica por qué el procedimiento puede ejecutarse dos veces sin duplicar resultados.

---
# M8_8.33 · Flujo reproducible completo

## 18. Diseño del flujo

El flujo completo separa responsabilidades:

```text
cargar → validar → transformar → seleccionar → resumir
→ representar → guardar → registrar → comprobar
```

La función coordinadora no debe contener todos los detalles. Llama a funciones pequeñas que ya pueden revisarse y probarse.

## 19. Función de carga

La ruta de la base se recibe como argumento. La función devuelve un DataFrame y no crea resultados adicionales.

In [ ]:
def cargar_datos_sqlite(ruta_base_datos):
    """Carga mediciones y metadatos de pozos desde SQLite."""

    ruta_base_datos = Path(ruta_base_datos)

    if not ruta_base_datos.exists():
        raise FileNotFoundError(
            f"No existe la base de datos: {ruta_base_datos}"
        )

    consulta = """
    SELECT
        m.id_pozo,
        m.fecha,
        p.acuifero,
        p.cota_terreno_m,
        m.profundidad_nivel_m,
        m.precipitacion_mm,
        m.conductividad_uScm
    FROM mediciones AS m
    INNER JOIN pozos AS p
        ON m.id_pozo = p.id_pozo
    ORDER BY m.id_pozo, m.fecha;
    """

    with sqlite3.connect(ruta_base_datos) as conexion:
        datos_cargados = pd.read_sql_query(
            consulta,
            conexion,
            parse_dates=["fecha"]
        )

    return datos_cargados

## 20. Registro de entorno

Para repetir una ejecución conviene conservar fecha, versiones, configuración y validación. La fecha registra cuándo se ejecutó; no hace determinista el análisis.

In [ ]:
def crear_registro_ejecucion(configuracion, validacion):
    """Crea un diccionario con información básica de trazabilidad."""

    registro = {
        "fecha_utc": datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),
        "python": sys.version.split()[0],
        "sistema_operativo": platform.system(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "configuracion": configuracion,
        "validacion": validacion
    }

    return registro

## 21. Manifiesto de resultados

Un **manifiesto** enumera los archivos producidos. Permite verificar qué entregables generó el flujo y dónde están.

In [ ]:
def crear_manifiesto(rutas_resultados, raiz):
    """Devuelve rutas relativas y tamaños de los archivos generados."""

    raiz = Path(raiz)
    filas = []

    for ruta in rutas_resultados:
        ruta = Path(ruta)

        filas.append({
            "archivo": str(ruta.relative_to(raiz)),
            "tamaño_bytes": ruta.stat().st_size
        })

    manifiesto = pd.DataFrame(filas)
    return manifiesto

## 22. Función coordinadora

La función siguiente integra el flujo. Los pasos permanecen separados y los archivos generados se recopilan explícitamente.

Que el flujo termine sin errores significa que las instrucciones se ejecutaron. No demuestra que las decisiones científicas sean adecuadas.

In [ ]:
def ejecutar_flujo(ruta_base_datos, configuracion, carpeta_salida):
    """Ejecuta el análisis reproducible y devuelve resultados y rutas."""

    carpeta_salida = Path(carpeta_salida)
    carpeta_tablas = carpeta_salida / "tables"
    carpeta_figuras = carpeta_salida / "figures"
    carpeta_logs = carpeta_salida / "logs"

    for carpeta in [carpeta_tablas, carpeta_figuras, carpeta_logs]:
        carpeta.mkdir(parents=True, exist_ok=True)

    logger.info("Inicio del flujo completo")

    datos_flujo = cargar_datos_sqlite(ruta_base_datos)
    logger.info("Datos cargados: %s filas", len(datos_flujo))

    validacion = validar_mediciones(datos_flujo)
    logger.info("Validación completada")

    resultados = []
    rutas_generadas = []

    for id_pozo in configuracion["pozos"]:
        try:
            datos_pozo = seleccionar_pozo(datos_flujo, id_pozo)

            resumen_pozo = resumir_variable(
                datos_entrada=datos_pozo,
                variable=configuracion["variable"]
            )

            resumen_pozo["id_pozo"] = id_pozo
            resultados.append(resumen_pozo)

            extension = configuracion["formato_figura"]
            ruta_figura = (
                carpeta_figuras
                / f"{id_pozo}_profundidad_nivel.{extension}"
            )

            guardar_serie_temporal(
                datos_pozo=datos_pozo,
                ruta=ruta_figura,
                dpi=configuracion["dpi"]
            )

            rutas_generadas.append(ruta_figura)
            logger.info("Pozo procesado: %s", id_pozo)

        except ValueError as error:
            logger.error("No se pudo procesar %s: %s", id_pozo, error)

    resumen_final = pd.DataFrame(resultados)
    resumen_final = resumen_final.set_index("id_pozo")

    ruta_tabla = carpeta_tablas / "resumen_pozos.csv"
    guardar_dataframe_csv(resumen_final, ruta_tabla)
    rutas_generadas.append(ruta_tabla)

    registro = crear_registro_ejecucion(
        configuracion=configuracion,
        validacion=validacion
    )

    ruta_registro = carpeta_logs / "registro_ejecucion.json"
    ruta_registro.write_text(
        json.dumps(registro, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    rutas_generadas.append(ruta_registro)

    manifiesto = crear_manifiesto(
        rutas_resultados=rutas_generadas,
        raiz=carpeta_salida
    )

    ruta_manifiesto = carpeta_logs / "manifiesto_resultados.csv"
    manifiesto.to_csv(ruta_manifiesto, index=False)
    rutas_generadas.append(ruta_manifiesto)

    logger.info("Flujo completado")

    return resumen_final, validacion, manifiesto

## 23. Ejecutar el flujo

Se utiliza `data/output`, la misma carpeta de resultados definida desde el inicio del repositorio.

In [ ]:
resumen_final, validacion_final, manifiesto_final = ejecutar_flujo(
    ruta_base_datos=ruta_base_datos,
    configuracion=configuracion_leida,
    carpeta_salida=OUTPUT
)

print("Resumen")
display(resumen_final)

print("Validación")
print(validacion_final)

print("Manifiesto")
display(manifiesto_final)

## 24. Comprobaciones posteriores

Las comprobaciones posteriores verifican que los entregables existen y contienen la estructura esperada. No sustituyen la interpretación científica.

In [ ]:
ruta_tabla_final = TABLES / "resumen_pozos.csv"
ruta_registro_final = LOGS / "registro_ejecucion.json"
ruta_manifiesto_final = LOGS / "manifiesto_resultados.csv"

assert ruta_tabla_final.exists()
assert ruta_registro_final.exists()
assert ruta_manifiesto_final.exists()
assert len(resumen_final) > 0
assert resumen_final.index.is_unique

print("Comprobaciones de archivos y estructura superadas.")

## 25. Qué hace reproducible este flujo

- utiliza los datos compartidos en SQLite;
- conserva los originales;
- recibe rutas y parámetros;
- separa cálculo y entrada/salida;
- valida antes de analizar;
- registra versiones y configuración;
- produce nombres de archivo previsibles;
- genera un manifiesto;
- puede volver a ejecutarse;
- incluye comprobaciones automáticas.

Aspectos que todavía requieren juicio humano:

- pertinencia de la variable;
- tratamiento de ausentes y extremos;
- comparabilidad entre pozos;
- dependencia espacial y temporal;
- interpretación hidrogeológica.

## 26. Continuidad con M8_8.19–27

- **M8_8.19–21:** lógica, tipos, funciones, errores, entornos y Git.
- **M8_8.22–24:** arrays, DataFrames, archivos y figuras.
- **M8_8.25–27:** SQLite, descripción, calidad, correlación y dependencia.
- **M8_8.31–33:** conversión de esos pasos en un procedimiento reutilizable y trazable.

La automatización reutiliza conocimientos anteriores. No sustituye la comprensión de datos, estadística o hidrogeología.

### Actividad integradora M8_8.33

Adapta el flujo a otra selección de pozos o variable.

Entregables:

1. configuración utilizada;
2. tabla de resultados;
3. una figura por pozo;
4. registro de ejecución;
5. manifiesto de resultados;
6. texto breve con patrón observado y una limitación;
7. al menos dos comprobaciones automáticas.

La entrega debe poder regenerarse ejecutando el notebook desde el principio.

## Síntesis

- Una función expresa una responsabilidad y devuelve un resultado reutilizable.
- La validación protege el flujo antes del cálculo.
- La automatización coordina funciones conocidas sobre varias unidades.
- La configuración separa decisiones variables de la lógica estable.
- Los logs registran lo ocurrido.
- La idempotencia facilita repetir ejecuciones.
- El manifiesto enumera los resultados.
- Un flujo técnicamente reproducible sigue necesitando interpretación científica.